### Learning Instruction Finetuning - LORA

In [ ]:
%load_ext autoreload
%load_ext tensorboard
%autoreload 2

In [ ]:
from pathlib import Path
import os
import socket
import sys
from dotenv import load_dotenv
from pathlib import Path
import sys

In [ ]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()

sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")
load_dotenv((project_root /'env.env').resolve())
data_dir = project_root / "datasets"

Project Root: /Users/vedaangchopra/all_data/complete_technical_work/reference_and_learning_content/deep_learning


In [ ]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "pace" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

False

In [ ]:
import torch
from datasets import Dataset as hf_data
from torch.utils.data import Dataset
import tiktoken
import numpy, tqdm
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'mps'

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [ ]:

from model_training.framework.data.post_train.instruction_tuning.raw_data.simple_example.instruct_data import download_and_load_file
from model_training.framework.data.post_train.instruction_tuning.prompt_style.styles import SFTDatasetPrepare, AlpacaFormatOld
from model_training.framework.data.post_train.dataloader import InstructionDataset, custom_collate_fn, create_instruction_dataloader

    ## Fix this import
    
    # raise Exception("Not Running on Pace")
        # from src.model import GPT

In [ ]:
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(i, props.name, f"{props.total_memory / 1024**3:.1f} GiB")

GPU count: 0


### Step-1:-  Preparing the dataset

We are using Alpaca Format, every models have their own format, so the format of the tuning can differ. <br>
THis format is called ALPACA format{instruction, input output}... <br>
 There are different formats that exist and the data format doesn't matter, we need to take prompt style templates for instruction tuning. We need to apply those prompt templates for instruction tuning. 

In [ ]:
file_name = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(data_dir=(data_dir / "instruction_tuning"), file_name=file_name, url = url)
print("Number of entries:", len(data))
data[50] , data[999]

Number of entries: 1100


({'instruction': 'Identify the correct spelling of the following word.',
  'input': 'Ocassion',
  'output': "The correct spelling is 'Occasion.'"},
 {'instruction': "What is an antonym of 'complicated'?",
  'input': '',
  'output': "An antonym of 'complicated' is 'simple'."})

In [ ]:
# alpaca = AlpacaFormatOld(data)

In [ ]:
# ## Example of format
# print(alpaca.format_input(instruction=data[50]['instruction'], in_text=data[50]['input']))
# print("=============================================")
# print(alpaca.detailed_response(out_text = data[50]['output']))

In [ ]:
# sample = (alpaca.complete_response(instruction=data[999]['instruction'], in_text=data[999]['input'], out_text=data[999]['output']))
# print(sample)

In [ ]:
dataset  = hf_data.from_list(data)
sft_data= SFTDatasetPrepare(model_name=MODEL_NAME, template="model",)
prepared = sft_data.prepare(dataset)

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

In [ ]:
print(prepared[0])
print(sft_data.preview(prepared[0]))

{'prompt': [{'content': 'Evaluate the following phrase by transforming it into the spelling given.\n\nInput:\nfreind --> friend', 'role': 'user'}], 'completion': [{'content': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".', 'role': 'assistant'}]}
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Evaluate the following phrase by transforming it into the spelling given.

Input:
freind --> friend<|im_end|>
<|im_start|>assistant
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".<|im_end|>



#### Spliting the data 

In [ ]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = prepared.select(range(train_portion))
test_data = prepared.select(
    range(train_portion, train_portion + test_portion)
)
val_data = prepared.select(
    range(train_portion + test_portion, len(prepared))
)

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


### Step-2: - Prepare the Training Set Batches

#### Setting up Tokenizer

In [ ]:
# tokenizer = tiktoken.get_encoding("gpt2")
# pad_token_id= tokenizer.encode('<|endoftext|>', allowed_special={"<|endoftext|>"})[0]
# pad_token_id
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

pad_token_id = tokenizer.pad_token_id
pad_token_id

151643

In [ ]:
train_dataset = InstructionDataset(
    ds=train_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)


In [ ]:
val_dataset = InstructionDataset(
    ds=val_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)
test_dataset = InstructionDataset(
    ds=test_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)

#### Sample Data Prepare

In [ ]:

inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
custom_collate_fn(batch, pad_token_id=pad_token_id)

(tensor([[     0,      1,      2,      3,      4],
         [     5,      6, 151643, 151643, 151643],
         [     7,      8,      9, 151643, 151643]]),
 tensor([[     1,      2,      3,      4, 151643],
         [     6, 151643,   -100,   -100,   -100],
         [     8,      9, 151643,   -100,   -100]]))

In [ ]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],  # 1st training example
     [-0.5, 1.5]]  # 2nd training example
)
targets_1 = torch.tensor([0, 1])


loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # New 3rd training example
)
targets_2 = torch.tensor([0, 1, 1])

loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)
targets_3 = torch.tensor([0, 1, -100])

loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

tensor(1.1269)
tensor(0.7936)
tensor(1.1269)
loss_1 == loss_3: tensor(True)


### Creating, Training and Testing Dataset

In [ ]:
from functools import partial
# Changing the defaults at runtime
custom_collate_fn1 = partial(custom_collate_fn, device=device, allowed_max_length = 1024, pad_token_id=pad_token_id)

In [ ]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = create_instruction_dataloader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)
val_loader = create_instruction_dataloader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_loader = create_instruction_dataloader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [ ]:
print("Train loader:")
for inputs, targets in train_loader:
    sample_input , sample_target = inputs, targets
    print(inputs.shape, targets.shape)
    break

Train loader:
torch.Size([8, 59]) torch.Size([8, 59])


### Step-4: - Loading the Model Testing inference

In [ ]:
from model_training.framework.model.load_model.gpt2.model import return_model
from model_training.framework.model.load_model.gpt2.configs import load_model_config
from model_training.framework.model.components.inference.inference import gen_sample
from model_training.framework.model.tokenizer.tokenizer_utils import text_to_token_ids, token_ids_to_text